In [16]:
import sys, types
import pandas as pd
import json
sys.modules['audioop'] = types.ModuleType('audioop')


In [6]:
sys.path.append('../')
from src.extrair_audio import transcrever_youtube_yt_dlp, transcrever_youtube_yt_dlp_quebrado

In [7]:
from dotenv import load_dotenv
load_dotenv()
import os

# Código teste de chamada ao GROQ

In [8]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": "Olá! Apenas testando a API da Groq."}
    ]
)

print(response.choices[0].message.content)


Olá! Estou aqui para ajudar. A API da Groq é uma API de processamento de linguagem natural (NLP) que fornece uma variedade de funcionalidades para analisar e processar texto. Se você tiver alguma pergunta ou precisar de ajuda com algo específico, sinta-se à vontade para perguntar!

Qual é a sua pergunta ou qual é o seu objetivos ao usar a API da Groq?


# Transcrição da receita

In [10]:
TEXTO_TRANSCRICAO = pd.read_csv('../data/transcricao_receita.txt', delimiter=";")

# Prompt de receita

In [13]:
prompt_sistema = """
  <SYSTEM>
  <PERSONA>
  Você é um chef de cozinha profissional, especialista em culinária internacional e criação de receitas detalhadas e muito bem explicadas. 
  Você escreve sempre de forma clara, organizada e acessível.
  </PERSONA>

  <TAREFA>
  Gerar receitas completas, detalhadas e otimizadas para leitura.
  </TAREFA>

  <INSTRUCOES>
  - Sempre siga exatamente a transcrição fornecida pelo usuário.
  - Nunca invente informações que contradigam a transcrição.
  - A receita final deve ser altamente estruturada.
  - Sempre responder em JSON válido.
  </INSTRUCOES>

  <FORMATO_RESPOSTA>
  A resposta deve ser estritamente neste formato:

  {
    "nome_receita": "",
    "descricao": "",
    "ingredientes": [],
    "modo_preparo": [],
    "tempo_preparo": "",
    "rendimento": "",
    "observacoes": ""
  }
  </FORMATO_RESPOSTA>
  </SYSTEM>
  """

prompt_usuario = f"""

<TRANSCRIÇÃO A SEGUIR>
{TEXTO_TRANSCRICAO}
"""

In [14]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": prompt_usuario}
    ]
)

resultado = response.choices[0].message.content
print(resultado)

Aqui está a receita do Zabaione em JSON:

{
  "nome_receita": "Zabaione",
  "descricao": "Um doce italiano maravilhoso feito com gemas, açúcar e vinho licoroso.",
  "ingredientes": [
    { "nome": "Gemadas", "quantidade": "6" },
    { "nome": "Açúcar", "quantidade": "40g" },
    { "nome": "Vinho licoroso (Marsala)", "quantidade": "2 colheres de sopa" },
    { "nome": "Enfeitinhos (opcional)", "quantidade": "vários" }
  ],
  "modo_preparo": [
    {
      "passo": "Ligue o banho Maria e coloque a panela com água até atingir a temperatura desejada.",
      "instruções": ""
    },
    {
      "passo": "Coloque as gemas e o açúcar na panela e bata com um fouet até que as gemas estejam bem misturadas e aeradas.",
      "instruções": ""
    },
    {
      "passo": "Leve a panela ao fogo baixo e cozinhe por 10 minutos, mexendo ocasionalmente, até que a mistura comece a espessar.",
      "instruções": ""
    },
    {
      "passo": "Coloque o vinho licoroso na panela e misture bem.",
      "ins

In [18]:
resposta = resultado  # texto retornado pelo Groq

# Extrair o JSON — caso venha com texto antes/depois, capture apenas o JSON
inicio = resposta.find("{")
fim = resposta.rfind("}") + 1
json_puro = resposta[inicio:fim]

dados = json.loads(json_puro)

print(dados)  # agora é um dicionário Python

# Se quiser transformar em DataFrame:
df_receita = pd.json_normalize(dados)
df_receita.head()

{'nome_receita': 'Zabaione', 'descricao': 'Um doce italiano maravilhoso feito com gemas, açúcar e vinho licoroso.', 'ingredientes': [{'nome': 'Gemadas', 'quantidade': '6'}, {'nome': 'Açúcar', 'quantidade': '40g'}, {'nome': 'Vinho licoroso (Marsala)', 'quantidade': '2 colheres de sopa'}, {'nome': 'Enfeitinhos (opcional)', 'quantidade': 'vários'}], 'modo_preparo': [{'passo': 'Ligue o banho Maria e coloque a panela com água até atingir a temperatura desejada.', 'instruções': ''}, {'passo': 'Coloque as gemas e o açúcar na panela e bata com um fouet até que as gemas estejam bem misturadas e aeradas.', 'instruções': ''}, {'passo': 'Leve a panela ao fogo baixo e cozinhe por 10 minutos, mexendo ocasionalmente, até que a mistura comece a espessar.', 'instruções': ''}, {'passo': 'Coloque o vinho licoroso na panela e misture bem.', 'instruções': ''}, {'passo': 'Continue cozinando por mais 5 minutos, até que a mistura atinja a consistência desejada.', 'instruções': ''}, {'passo': 'Retire a panela 

,nome_receita,descricao,ingredientes,modo_preparo,tempo_preparo,rendimento,observacoes
0,Zabaione,"Um doce italiano maravilhoso feito com gemas, ...","[{'nome': 'Gemadas', 'quantidade': '6'}, {'nom...",[{'passo': 'Ligue o banho Maria e coloque a pa...,1 hora 20 minutos,6 porções,O Zabaione é um doce italiano tradicional que ...


Trechos

> dividir em função que baixa o audio e outra que transcreve

> testar outros modelos whisper e avaliar desempenho (tempo x qualidade texto)

> extrator vai gerar uma lista de trechos de áudio. 
> Depois, gerar uma lista de transcrições da lista dos audios
> Agregar as transcrições
> Testar conexão com o Grok usando prompt genérico
> Depois prompt específico do próprio GPT para ver resultados dos 2 tipos de vídeo